# 00 — Zero-shot FRAME baseline (Qwen3-VL-8B)

The always-valid floor: an open VLM answering FRAME questions with **no fine-tuning**, scored by the official `focus.Evaluator`. One frame is sampled per question at its timestamp and passed as an image. All logic lives in the `frame` library (`src/frame/`); this notebook only supplies config and launches the run.

**Single variable:** none (this establishes the baseline). Runs on GPU (RunPod A100/L40S). Set `FRAME_N_EVAL` in the environment to cap questions for a SMOKE/sample run; leave unset for the full test set.

In [ ]:
# ── bootstrap ─────────────────────────────────────────────────────
import os, sys, logging
from pathlib import Path

SRC = os.environ.get("FRAME_SRC", str(Path.cwd().parents[1] / "src"))
if SRC not in sys.path:
    sys.path.insert(0, SRC)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s", datefmt="%H:%M:%S")

from frame import BaselineConfig, run_baseline

In [ ]:
# ── config (inline) ───────────────────────────────────────────────
_n = os.environ.get("FRAME_N_EVAL")
cfg = BaselineConfig(
    n_eval=(int(_n) if _n and int(_n) > 0 else None),
    run_name=os.environ.get("FRAME_RUN_NAME", "00_zeroshot_qwen3vl"),
)
print(cfg)

In [ ]:
# ── launch ────────────────────────────────────────────────────────
report = run_baseline(cfg)

print("\n================ FRAME BASELINE REPORT ================")
print(f"pre_evaluation_score : {report['pre_evaluation_score']}")
print(f"overall mean acc     : {report['overall_mean_accuracy']}")
print(f"raw accuracy         : {report['raw_accuracy']}")
print(f"timed out (>5s)      : {report['n_timed_out']}")
print(f"latency p50/p95/p99  : {report['latency_s']['p50']:.2f} / {report['latency_s']['p95']:.2f} / {report['latency_s']['p99']:.2f} s")
print("by answer_format     :")
for k, v in report['by_answer_format'].items():
    print(f"    {k:16s} {v['accuracy']:.3f}  (n={v['count']})")